# Medallion Architecture Diagram

                    RAW SALES CSV
                         │
                         ▼
              ┌─────────────────────┐
              │       BRONZE        │
              │ dev.bronze.sales_raw│
              │                     │
              │ Raw data +          │
              │ ingestion_timestamp │
              └──────────┬──────────┘
                         │
                         ▼
              ┌─────────────────────┐
              │       SILVER        │
              │dev.silver.sales_clean│
              │                     │
              │ • Remove duplicates │
              │ • Fix data types    │
              │ • Remove invalid    │
              │   records           │
              └──────────┬──────────┘
                         │
                         ▼
              ┌─────────────────────┐
              │        GOLD         │
              │dev.gold.daily_revenue│
              │                     │
              │ • Daily revenue     │
              │ • Total orders      │
              │ • Quantity sold     │
              │ • Total discount    │
              └──────────┬──────────┘
                         │
                         ▼
                  ANALYSTS / BI

# Task 4 — Medallion Architecture

The solution follows the Medallion Architecture with three layers: Bronze, Silver, and Gold.

## Data Flow

Raw Sales CSV
    ↓
Bronze Layer
    ↓
Silver Layer
    ↓
Gold Layer
    ↓
Business Analytics

### Bronze Layer

Table: `dev.bronze.sales_raw1`

The Bronze layer stores the raw sales data as-is and adds an ingestion timestamp.

### Silver Layer

Table: `dev.silver.sales_clean1`

The Silver layer cleans the Bronze data by removing duplicates, fixing data types, and removing clearly invalid records.

### Gold Layer

Table: `dev.gold.daily_revenue`

The Gold layer contains business-ready daily revenue aggregations including total orders, quantity sold, discounts, and revenue.

## Architecture

Raw Sales CSV
       ↓
dev.bronze.sales_raw
       ↓
dev.silver.sales_clean
       ↓
dev.gold.daily_revenue
       ↓
Business Analytics

# Task 5 — Lakeflow Designer

A Silver-layer transformation was recreated using Lakeflow Designer's visual interface.

## Transformation Recreated

The Bronze table `dev.bronze.sales_raw` was used as the source.

A filter transformation was applied to keep only records where:

`quantity > 0`

## Code-Based Transformation

The same transformation was previously implemented using PySpark:

```python
silver_df = (
    silver_df
    .filter(col("quantity").isNotNull())
    .filter(col("quantity") > 0)
)

![](/Workspace/Data_Engineering_Assignments/Assignments/Assignment6/runing pipeline.png)

# Task 6 — Lakeflow Job

A Lakeflow Job was created to orchestrate the Bronze, Silver, and Gold processing notebooks.

## Job Name

`Day6_Medallion_Pipeline`

## Tasks

1. **Bronze**
   - Notebook: `Day6_Bronze`
   - Creates/loads `dev.bronze.sales_raw`

2. **Silver**
   - Notebook: `Day6_Silver`
   - Depends on: `Bronze`
   - Creates `dev.silver.sales_clean`

3. **Gold**
   - Notebook: `Day6_Gold`
   - Depends on: `Silver`
   - Creates `dev.gold.daily_revenue`

## Dependency Flow

Bronze → Silver → Gold

The dependencies ensure that the Silver task runs only after Bronze completes successfully, and Gold runs only after Silver completes successfully.

## Schedule

The job is configured with a daily schedule to automate the Bronze → Silver → Gold processing workflow.

## Validation

The job was manually triggered using "Run now" to validate the complete dependency chain.

#  **Advanced Tasks** 

# Task 8 — Data Plane vs Control Plane

## Control Plane

The Control Plane manages and coordinates the data processing workflow.

In this solution, the Lakeflow/Databricks Job acts as the orchestration layer. It is responsible for scheduling the pipeline, managing task dependencies, and monitoring task execution.

The workflow is:

Bronze → Silver → Gold

The Control Plane determines when each task should run and ensures that dependent tasks run only after their upstream tasks complete successfully.

## Data Plane

The Data Plane is responsible for the actual data processing.

In this solution, the Bronze, Silver, and Gold notebooks perform the data processing operations.

The Data Plane performs the following operations:

Raw Sales CSV
    ↓
Bronze: Store raw data
    ↓
Silver: Clean and validate data
    ↓
Gold: Create business-ready aggregations

## Security and Governance

The solution uses Unity Catalog to organize and govern the Bronze, Silver, and Gold tables.

The tables are:

- `dev.bronze.sales_raw1`
- `dev.silver.sales_clean1`
- `dev.gold.daily_revenue`
- `dev.gold.product_performance`

Access can be controlled according to user roles and business requirements. Data consumers can be given access to appropriate Gold tables without necessarily receiving unrestricted access to the raw Bronze layer.

## Failure and Retry Considerations

The pipeline uses task dependencies:

Bronze → Silver → Gold

If the Bronze task fails, the Silver and Gold tasks should not run because they depend on successful upstream processing.

If Silver fails, Gold should not run because the Gold layer depends on successfully cleaned Silver data.

Failed tasks can be configured with retry policies in the Job. After a successful retry, downstream tasks can continue according to their dependencies.

This dependency structure helps prevent incomplete or invalid data from reaching the Gold layer.

                         CONTROL PLANE
                              │
                              │
                    ┌─────────▼─────────┐
                    │   Lakeflow Job    │
                    │                   │
                    │ • Schedule        │
                    │ • Dependencies    │
                    │ • Monitoring      │
                    └─────────┬─────────┘
                              │
                              │ triggers
                              ▼
                         DATA PLANE
                              │
                    ┌─────────▼─────────┐
                    │ Bronze Notebook   │
                    └─────────┬─────────┘
                              │
                              ▼
                    ┌───────────────────┐
                    │ Silver Notebook   │
                    └─────────┬─────────┘
                              │
                              ▼
                    ┌───────────────────┐
                    │ Gold Notebook     │
                    └───────────────────┘

# Task 9 — Gold Analysis and Data Quality Trace

## Part A — Business Analysis

Business Question:

Which dates generated the highest revenue?

The `dev.gold.daily_revenue` table will be used to answer this question.

This Gold table provides daily revenue metrics that can be consumed directly by business users and analysts.

In [0]:
from pyspark.sql.functions import col

daily_revenue_df = spark.table("dev.gold.daily_revenue")

top_revenue_days = (
    daily_revenue_df
    .orderBy(col("total_revenue").desc())
)

display(top_revenue_days)

### Data Quality Trace

## Part B — Data Quality Trace

A data-quality issue was traced from the Bronze layer to the Silver layer.

### Issue Identified

An invalid sales record was found in the Bronze table where the quantity was less than or equal to zero.

### Bronze

The record was present in:

`dev.bronze.sales_raw1`

Example:

`order_id = None`

The record contained an invalid quantity.

### Silver

The Silver transformation contains the validation rule:

`quantity > 0`

Therefore, the invalid record was removed during Silver processing.

The same `order_id` was checked in:

`dev.silver.sales_clean`

and was not present.

### Gold Impact

Because the invalid record was removed in the Silver layer, it was not included in the Gold aggregations.

This demonstrates how data-quality rules in the Silver layer protect business-facing Gold reports from invalid source records.

## Task 9 Summary

### Business Analysis

The Gold table `dev.gold.daily_revenue` was used to identify the highest-revenue dates. A visualization was created to make the daily revenue trend easier for business users to understand.

### Data Quality Trace

An invalid record was identified in the Bronze layer and traced through the Silver transformation. The Silver validation rule removed the invalid record before it could affect Gold-level business reporting.

This demonstrates the value of the Medallion Architecture: Bronze preserves the source data, Silver applies data-quality rules, and Gold provides trusted business-ready information.